# Solution 06 on Google Colab — VLM LoRA fine-tune, synthetic augmented (tuned upsampling + LLM)

**Solution 05 plus method (c) (LLM-assisted injection) included.** A genuinely separate solution
folder from solution 05 -- not a config toggle within it -- specifically so the two can run in
**two parallel Colab sessions** without ever touching the same Drive paths. Reuses solution 05's
`generate_synthetic.py` directly (same `--max_upsample_multiplier` fix, same tuned rarity multipliers), so the only real difference between solutions 05 and 06 is whether method (c) ran -- a clean, single-variable comparison.

**Before running anything:** `Runtime > Change runtime type > T4/L4/A100 GPU`.

**Assumption about your Drive layout:** same as the other solutions -- `solutions/`, `participant_kit/`, `data/`, `shroom-vis-images/` all live together under one folder on your Drive, and **solutions 04 and/or 05 have already been run at least once** (this notebook reuses their `ocr_cache.json` automatically -- OCR grounding doesn't change across any of these solutions; solution 03's cache, if present, is an equally valid fallback too).

**To run solutions 05 and 06 in parallel**: open both notebooks in separate tabs, connect each to its own runtime (see the earlier discussion on setting different accelerators per session), and just run each normally -- being separate folders with separate `CHECKPOINT_DIR`s means there's nothing to configure to keep them from colliding; it's impossible by construction.

**Checkpointing, resume, sequence-length OOM protection**: identical to solutions 04/05's notebooks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- Config: edit these ---
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SHROOM"   # <-- change to wherever your project actually lives on Drive
LOCAL_PROJECT_PATH = "/content/SHROOM"
GPU_INDEX = 0        # Colab only ever exposes one GPU -- leave this at 0
EPOCHS = 2
GRAD_ACCUM = 8
SAVE_STEPS = 250     # full-run checkpoint frequency (optimizer steps)
MAX_UPSAMPLE_MULTIPLIER = 3   # keep this equal to whatever solution 05 actually used, so the
                               # only real difference between the two runs is method (c)

sol06_local = f"{LOCAL_PROJECT_PATH}/solutions/06_vlm_lora_synthetic_tuned_llm"

# Own Drive path -- entirely separate from solution 05's, so running both in parallel is safe
# by construction (no shared writable path between them at all).
CHECKPOINT_DIR = f"{DRIVE_PROJECT_PATH}/solutions/06_vlm_lora_synthetic_tuned_llm/checkpoint"

# This solution is fixed to SKIP_LLM=False by design -- it's the whole point of this being a
# separate solution from 05. A fresh Anthropic key is required (or reuse solution 04's/06's own
# existing llm_synth_cache.json, handled automatically below -- if that already covers what you
# need, you may not need a key at all for a re-run).
SKIP_LLM = False
# Never hardcode a real key here. Set it as a Colab secret (key icon in the left sidebar, named
# ANTHROPIC_API_KEY, "Notebook access" enabled) or paste one into the input() prompt below at
# runtime -- either way it never gets written into this file.
if "ANTHROPIC_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("DRIVE_PROJECT_PATH:", DRIVE_PROJECT_PATH)
print("LOCAL_PROJECT_PATH:", LOCAL_PROJECT_PATH)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("MAX_UPSAMPLE_MULTIPLIER:", MAX_UPSAMPLE_MULTIPLIER)
print("SKIP_LLM:", SKIP_LLM)

In [ ]:
# Copy just the subfolders the pipeline actually needs from Drive to local disk.
os.makedirs(LOCAL_PROJECT_PATH, exist_ok=True)
for sub in ["data", "shroom-vis-images", "participant_kit", "solutions"]:
    src = f"{DRIVE_PROJECT_PATH}/{sub}"
    dst = f"{LOCAL_PROJECT_PATH}/{sub}"
    if os.path.exists(dst) and os.listdir(dst):
        print(f"skip (already present locally, {len(os.listdir(dst))} entries): {sub}")
        continue
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"{src} does not exist on your Drive. Check DRIVE_PROJECT_PATH and the exact folder "
            f"name/casing on your Drive (run `!ls \"{DRIVE_PROJECT_PATH}\"` in a new cell to see "
            f"what's actually there)."
        )
    print(f"copying {sub} from Drive ... (this can take a few minutes for shroom-vis-images/)")
    !cp -r "{src}" "{dst}"
    n = len(os.listdir(dst)) if os.path.exists(dst) else 0
    if n == 0:
        raise RuntimeError(
            f"Copy of {sub} produced an empty directory at {dst} -- something went wrong "
            f"(check the cp output above for errors, e.g. permission issues or a Drive timeout)."
        )
    print(f"  -> {n} entries copied")
print("done")
!ls "{LOCAL_PROJECT_PATH}"

In [ ]:
!pip install -q "git+https://github.com/huggingface/transformers"
!pip install -q "huggingface-hub<1.0" "accelerate>=0.34" "peft>=0.13" sentencepiece
!pip install -q bitsandbytes anthropic easyocr sacremoses

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
if (major, minor) < (2, 6):
    print("\ntorch is older than 2.6 -- upgrading (see solution 02's notebook for why):")
    !pip install -q -U torch torchvision torchaudio
    print("IMPORTANT: Runtime > Restart runtime now, then re-run this notebook from the top.")
else:
    print("torch version OK.")

In [ ]:
%cd {LOCAL_PROJECT_PATH}
!python -m solutions.common.scorer_local

### Reuse solutions 04/05's caches

OCR grounding is identical across solutions 03, 04, 05, and 06. LLM-assisted injection targets are
identical between solutions 04 and 06 too (both include method (c) with the same caps). These
cells check solution 06's own Drive path first, then fall back to solution 05's, then solution
04's, then solution 03's -- so nothing gets rebuilt or re-spent if any prior run already produced it.

In [ ]:
import shutil

ocr_cache_drive_06 = f"{DRIVE_PROJECT_PATH}/solutions/06_vlm_lora_synthetic_tuned_llm/ocr_cache.json"
ocr_cache_drive_05 = f"{DRIVE_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned/ocr_cache.json"
ocr_cache_drive_04 = f"{DRIVE_PROJECT_PATH}/solutions/04_vlm_lora_synthetic_augmented_llm/ocr_cache.json"
ocr_cache_drive_03 = f"{DRIVE_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented/ocr_cache.json"
ocr_cache_local = f"{sol06_local}/ocr_cache.json"

SKIP_OCR_EXTRACT = False
os.makedirs(sol06_local, exist_ok=True)
for label, drive_path in [("solution 06", ocr_cache_drive_06), ("solution 05", ocr_cache_drive_05), ("solution 04", ocr_cache_drive_04), ("solution 03", ocr_cache_drive_03)]:
    if os.path.exists(drive_path):
        print(f"Found {label}'s ocr_cache.json on Drive -- reusing it.")
        shutil.copy(drive_path, ocr_cache_local)
        SKIP_OCR_EXTRACT = True
        break
if not SKIP_OCR_EXTRACT:
    print("No existing OCR cache found on Drive for any solution -- this run will build one from scratch (slow, budget hours).")
print("SKIP_OCR_EXTRACT =", SKIP_OCR_EXTRACT)

In [ ]:
# LLM-assisted injection cache: same fallback chain (06 -> 04, since only 04/06 ever include
# method (c) -- solutions 03/05 never touch this file).
llm_cache_drive_06 = f"{DRIVE_PROJECT_PATH}/solutions/06_vlm_lora_synthetic_tuned_llm/llm_synth_cache.json"
llm_cache_drive_04 = f"{DRIVE_PROJECT_PATH}/solutions/04_vlm_lora_synthetic_augmented_llm/llm_synth_cache.json"
llm_cache_local = f"{sol06_local}/llm_synth_cache.json"

for label, drive_path in [("solution 06", llm_cache_drive_06), ("solution 04", llm_cache_drive_04)]:
    if os.path.exists(drive_path):
        print(f"Found {label}'s llm_synth_cache.json on Drive -- reusing it (avoids re-spending API calls).")
        shutil.copy(drive_path, llm_cache_local)
        break
else:
    print("No existing LLM cache found -- will call the API fresh.")

Smoke test next. Uses a local-only checkpoint dir (not `CHECKPOINT_DIR`/Drive) so it can never overwrite real full-run progress. Deliberately does not pass `--skip_ocr_extract` unless a cache was already found above.

In [ ]:
%cd {sol06_local}
skip_ocr_flag = "--skip_ocr_extract" if SKIP_OCR_EXTRACT else "

In [ ]:
!python run_pipeline.py --smoke_test --gpu_index {GPU_INDEX} --max_upsample_multiplier {MAX_UPSAMPLE_MULTIPLIER} {skip_ocr_flag}

In [ ]:
# Same cleanup as solutions 04/05's notebooks: clear a smoke-test-scoped ocr_cache.json before
# the full run so it rebuilds/reuses a real one instead.
if not SKIP_OCR_EXTRACT and os.path.exists(ocr_cache_local):
    os.remove(ocr_cache_local)
    print("Removed the smoke-test-scoped ocr_cache.json -- the full run below will build/reuse a real one.")
else:
    print("Nothing to clean up.")

Once the smoke test finishes cleanly, run the full pipeline. Same checkpointing/resume/
sequence-length protections as solutions 04/05's notebooks.

OOM fallback for a smaller GPU (T4/L4 instead of A100):
```
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} --max_upsample_multiplier {MAX_UPSAMPLE_MULTIPLIER} {skip_ocr_flag} --checkpoint_dir "{CHECKPOINT_DIR}" --save_steps {SAVE_STEPS} --resume --max_pixels 401408 --target_modules q_proj,k_proj,v_proj,o_proj
```

In [ ]:
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} --max_upsample_multiplier {MAX_UPSAMPLE_MULTIPLIER} {skip_ocr_flag} --checkpoint_dir "{CHECKPOINT_DIR}" --save_steps {SAVE_STEPS} --resume

In [ ]:
# Save the caches back to Drive (under solution 06's own path) in case either was built/extended fresh this run.
if os.path.exists(ocr_cache_local):
    os.makedirs(os.path.dirname(ocr_cache_drive_06), exist_ok=True)
    !cp "{ocr_cache_local}" "{ocr_cache_drive_06}"
    print("OCR cache saved to Drive (solution 06's own path).")
if os.path.exists(llm_cache_local):
    os.makedirs(os.path.dirname(llm_cache_drive_06), exist_ok=True)
    !cp "{llm_cache_local}" "{llm_cache_drive_06}"
    print("LLM-assisted injection cache saved to Drive (solution 06's own path).")

In [ ]:
# Copy the rest of the results back to Drive: predictions/scores, plus the augmented training set
# for provenance/debugging. checkpoint/ is NOT copied here -- the full-run cell above already saved
# it directly to CHECKPOINT_DIR on Drive throughout training.
dest = f"{DRIVE_PROJECT_PATH}/solutions/06_vlm_lora_synthetic_tuned_llm"
os.makedirs(dest, exist_ok=True)
for item in ["val_predictions.jsonl", "val_scores.json", "train_augmented.jsonl"]:
    src = f"{sol06_local}/{item}"
    if os.path.exists(src):
        print(f"copying {item} -> Drive ...")
        !cp -r "{src}" "{dest}/"
    else:
        print(f"skip (not found): {item}")
print("done -- check", dest, "in your Drive")

In [ ]:
import json
with open(f"{sol06_local}/val_scores.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Compare against solutions 02/04/05 before submitting anything

Reference points: solution 02 (`overall: iou=0.4380, cor=0.4712, cor_lbl=0.4172`), solution 04 (`overall: iou=0.4532, cor=0.4880, cor_lbl=0.4113`), and whatever solution 05 (no LLM, tuned upsampling) actually measured. The comparison that matters here is solution 05 vs. solution 06 -- since both share the same tuned multiplier, any difference between them isolates the effect of method (c) specifically.

## Test-set predictions (for SHROOM website submission)

Same as the other solutions: runs the trained checkpoint over the real, **unlabeled** `data/test.jsonl` (4,898 examples). Budget several hours (solution 04's own test run measured ~9.3s/example on an A100). `predict.py` has no periodic-save/resume support yet.

**Required before submitting**: split by language + format-check, same as the other solutions.

In [ ]:
test_predictions_path = f"{sol06_local}/test_predictions.jsonl"

predict_script = f"{LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data/predict.py"
test_file_path = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"

!python "{predict_script}" --checkpoint "{CHECKPOINT_DIR}" --val_file "{test_file_path}" --output "{test_predictions_path}" --gpu_index {GPU_INDEX}

In [ ]:
split_script = f"{LOCAL_PROJECT_PATH}/solutions/common/split_predictions_by_language.py"
!python "{split_script}" "{test_predictions_path}"

In [ ]:
format_checker_script = f"{LOCAL_PROJECT_PATH}/participant_kit/format_checker.py"
reference_file = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"
split_files = " ".join(
    f'"{sol06_local}/test_predictions.{lang}.jsonl"' for lang in ["en", "fr", "it", "zh"]
)
!python "{format_checker_script}" --reference-file "{reference_file}" {split_files}

In [ ]:
dest = f"{DRIVE_PROJECT_PATH}/solutions/06_vlm_lora_synthetic_tuned_llm"
os.makedirs(dest, exist_ok=True)
for lang in ["en", "fr", "it", "zh"]:
    src = f"{sol06_local}/test_predictions.{lang}.jsonl"
    if os.path.exists(src):
        print(f"copying test_predictions.{lang}.jsonl -> Drive ...")
        !cp "{src}" "{dest}/"
    else:
        print(f"skip (not found): test_predictions.{lang}.jsonl")
print("done -- check", dest, "in your Drive")

In [ ]:
import json

with open(f"{sol06_local}/test_predictions.en.jsonl") as f:
    lines = f.readlines()
print(f"{len(lines)} EN predictions written")
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))